# XRSUPP - RAG + LLM standalone

Notebook unico e modulare per testare tutta la pipeline senza frontend, senza FastAPI e senza database.

Pipeline inclusa:

- A) ingestione PDF da `pdfs/`
- B) pulizia testo
- C) chunking e salvataggio `index/chunks.jsonl`
- D) embedding con `sentence-transformers/all-MiniLM-L6-v2`
- E) indice FAISS `index/faiss.index`
- F) retrieval
- G) costruzione context + memoria breve/lunga
- H) chiamata LLM OpenAI-compatible, come nel backend XRSUPP/Ollama

Esecuzione batch da terminale:

```bash
cd modello
python3 -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
export OPENAI_BASE_URL=http://localhost:11434/v1
export LLM_MODEL=llama3.2
RAG_QUESTION="Quali funzionalita CRM descrivono i manuali?" jupyter nbconvert --to notebook --execute rag_llm_xrsupp.ipynb --inplace --ExecutePreprocessor.timeout=-1
```

Per ricreare indice e chunk da zero:

```bash
FORCE_REBUILD_INDEX=1 RAG_QUESTION="Domanda" jupyter nbconvert --to notebook --execute rag_llm_xrsupp.ipynb --inplace --ExecutePreprocessor.timeout=-1
```


In [1]:
# ============================================================
# 0) CONFIGURAZIONE E IMPORT
# ============================================================

from __future__ import annotations

import hashlib
import json
import os
import re
import uuid
from collections import deque
from pathlib import Path
from typing import Dict, List

import faiss
import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm


BASE_DIR = Path.cwd().resolve()
load_dotenv(BASE_DIR / ".env")

PDF_DIR = Path(os.getenv("PDF_DIR", BASE_DIR / "pdf_immagini")).resolve()
INDEX_DIR = Path(os.getenv("INDEX_DIR", BASE_DIR / "index")).resolve()
MEM_DIR = Path(os.getenv("MEM_DIR", BASE_DIR / "memory")).resolve()
OUTPUT_DEBUG_DIR = Path(os.getenv("OUTPUT_DEBUG_DIR", BASE_DIR / "outputs_debug")).resolve()

CHUNKS_PATH = INDEX_DIR / "chunks.jsonl"
FAISS_PATH = INDEX_DIR / "faiss.index"
INDEX_META_PATH = INDEX_DIR / "index_meta.json"
MEM_JSONL = MEM_DIR / "memory.jsonl"
MEM_FAISS = MEM_DIR / "memory.faiss"

EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2")
LLM_BASE_URL = os.getenv("OPENAI_BASE_URL", "http://localhost:11434/v1")
LLM_MODEL = os.getenv("LLM_MODEL", "llama3.2")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "ollama")

TOP_K = int(os.getenv("TOP_K", "12"))
RETRIEVAL_CANDIDATES = int(os.getenv("RETRIEVAL_CANDIDATES", "80"))
CONTEXT_MAX_CHARS = int(os.getenv("CONTEXT_MAX_CHARS", "16000"))
NEIGHBOR_CHUNKS = int(os.getenv("NEIGHBOR_CHUNKS", "1"))
MMR_LAMBDA = float(os.getenv("MMR_LAMBDA", "0.70"))
SESSION_ID = os.getenv("SESSION_ID", "demo_session_1")
FORCE_REBUILD_INDEX = os.getenv("FORCE_REBUILD_INDEX", "0") == "1"



ACTIVE_CORPUS_SIGNATURE = None

INDEX_DIR.mkdir(parents=True, exist_ok=True)
MEM_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DEBUG_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR:", BASE_DIR)
print("PDF_DIR:", PDF_DIR)
print("INDEX_DIR:", INDEX_DIR)
print("MEM_DIR:", MEM_DIR)
print("EMBEDDING_MODEL:", EMBEDDING_MODEL)
print("LLM:", LLM_MODEL, "@", LLM_BASE_URL)


BASE_DIR: /Users/matteoarutadm/lavoro/digital_mens/llm_yolo
PDF_DIR: /Users/matteoarutadm/lavoro/digital_mens/llm_yolo/pdf_immagini
INDEX_DIR: /Users/matteoarutadm/lavoro/digital_mens/llm_yolo/index
MEM_DIR: /Users/matteoarutadm/lavoro/digital_mens/llm_yolo/memory
EMBEDDING_MODEL: sentence-transformers/all-MiniLM-L6-v2
LLM: llama3.2 @ http://localhost:11434/v1


In [2]:
# ============================================================
# 1) MEMORIA BREVE E MEMORIA LUNGA VETTORIALE
# ============================================================

STM_MAX = 10
stm = deque(maxlen=STM_MAX)


def stm_add(role: str, content: str) -> None:
    stm.append({"role": role, "content": content})


def stm_text(max_chars: int = 1200) -> str:
    parts = []
    total = 0
    for message in list(stm)[-STM_MAX:]:
        line = f"{message['role'].upper()}: {message['content']}"
        if total + len(line) > max_chars:
            break
        parts.append(line)
        total += len(line) + 1
    return "\n".join(parts).strip()


class VectorMemory:
    """Memoria lunga conversazionale salvata su FAISS separato."""

    def __init__(self, embedder: SentenceTransformer):
        self.embedder = embedder
        self.items = []
        self.index = None

        if MEM_JSONL.exists():
            with MEM_JSONL.open("r", encoding="utf-8") as handle:
                for line in handle:
                    line = line.strip()
                    if line:
                        self.items.append(json.loads(line))

        if MEM_FAISS.exists():
            self.index = faiss.read_index(str(MEM_FAISS))

    def add_turn(self, session_id: str, user_text: str, assistant_text: str) -> None:
        text_for_embedding = f"USER: {user_text}\nASSISTANT: {assistant_text}"
        vec = self.embedder.encode([text_for_embedding], normalize_embeddings=True)
        vec = np.asarray(vec, dtype="float32")

        if self.index is None:
            self.index = faiss.IndexFlatIP(vec.shape[1])

        item = {
            "id": str(uuid.uuid4()),
            "session_id": session_id,
            "corpus_signature": ACTIVE_CORPUS_SIGNATURE,
            "user": user_text,
            "assistant": assistant_text,
            "embed_text": text_for_embedding,
        }

        self.index.add(vec)
        self.items.append(item)

        with MEM_JSONL.open("a", encoding="utf-8") as handle:
            handle.write(json.dumps(item, ensure_ascii=False) + "\n")
        faiss.write_index(self.index, str(MEM_FAISS))

    def search(self, session_id: str, query: str, top_k: int = 4) -> List[dict]:
        if self.index is None or not self.items:
            return []

        qv = self.embedder.encode([query], normalize_embeddings=True)
        qv = np.asarray(qv, dtype="float32")

        k = min(max(top_k * 5, top_k), len(self.items))
        scores, ids = self.index.search(qv, k)

        out = []
        for score, idx in zip(scores[0], ids[0]):
            if idx == -1:
                continue
            item = self.items[int(idx)]
            if item["session_id"] != session_id:
                continue
            if ACTIVE_CORPUS_SIGNATURE and item.get("corpus_signature") != ACTIVE_CORPUS_SIGNATURE:
                continue
            out.append({**item, "score": float(score)})
            if len(out) >= top_k:
                break
        return out


def ltm_text(mem_hits: List[dict], max_chars: int = 1200) -> str:
    parts = []
    total = 0
    for hit in mem_hits:
        line = f"[MEM score {hit['score']:.3f}] {hit['embed_text']}"
        if total + len(line) > max_chars:
            break
        parts.append(line)
        total += len(line) + 1
    return "\n".join(parts).strip()


In [3]:
# ============================================================
# A) INGESTIONE PDF, B) CLEANING, C) CHUNKING
# ============================================================

def clean_pdf_text(text: str) -> str:
    text = text or ""
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def read_pdf_pages(pdf_path: Path) -> List[str]:
    reader = PdfReader(str(pdf_path))
    pages = []
    for page in reader.pages:
        pages.append(clean_pdf_text(page.extract_text() or ""))
    return pages


def chunk_text(text: str, chunk_size: int = 1200, overlap: int = 180) -> List[str]:
    normalized = text.replace("\r\n", "\n").replace("\r", "\n")
    blocks = [block.strip() for block in re.split(r"\n{2,}", normalized) if block.strip()]
    chunks_out = []
    current = ""

    for block in blocks:
        candidate = block if not current else f"{current}\n\n{block}"
        if len(candidate) <= chunk_size:
            current = candidate
            continue

        if current:
            chunks_out.append(current)
            tail = current[-overlap:] if overlap > 0 else ""
            current = f"{tail}\n\n{block}".strip()
        else:
            start = 0
            while start < len(block):
                end = min(len(block), start + chunk_size)
                piece = block[start:end].strip()
                if piece:
                    chunks_out.append(piece)
                if end >= len(block):
                    current = ""
                    break
                start = max(end - overlap, start + 1)

        while len(current) > chunk_size:
            chunks_out.append(current[:chunk_size].strip())
            current = current[max(chunk_size - overlap, 1):].strip()

    if current:
        chunks_out.append(current)
    return [chunk for chunk in chunks_out if len(chunk.strip()) >= 50]


def ingest_pdfs(pdf_dir: Path = PDF_DIR, save_debug_text: bool = True) -> List[dict]:
    pdf_files = sorted(pdf_dir.rglob("*.pdf"))
    if not pdf_files:
        raise FileNotFoundError(f"Nessun PDF trovato in {pdf_dir}")

    all_chunks = []
    print("PDF trovati:", len(pdf_files))

    for pdf_path in tqdm(pdf_files, desc="Ingestione PDF"):
        source_name = str(pdf_path.relative_to(pdf_dir))
        pages = read_pdf_pages(pdf_path)

        if save_debug_text:
            debug_name = source_name.replace("/", "__")
            debug_path = OUTPUT_DEBUG_DIR / f"{Path(debug_name).stem}_all_pages.txt"
            with debug_path.open("w", encoding="utf-8") as out:
                for page_num, page_text in enumerate(pages, start=1):
                    out.write(page_text)
                    out.write("\n\n" + "-" * 40 + f" [FINE PAGINA {page_num}] " + "-" * 40 + "\n\n")

        for page_num, page_text in enumerate(pages, start=1):
            for chunk_index, chunk in enumerate(chunk_text(page_text)):
                all_chunks.append({
                    "id": f"{source_name}::p{page_num}::c{chunk_index}",
                    "source": source_name,
                    "page": page_num,
                    "chunk_index": chunk_index,
                    "text": chunk,
                })

    if not all_chunks:
        raise RuntimeError("Nessun chunk valido estratto dai PDF.")

    with CHUNKS_PATH.open("w", encoding="utf-8") as handle:
        for chunk in all_chunks:
            handle.write(json.dumps(chunk, ensure_ascii=False) + "\n")

    print("Creato:", CHUNKS_PATH)
    print("Chunk totali:", len(all_chunks))
    print("PDF indicizzati:", len({chunk["source"] for chunk in all_chunks}))
    return all_chunks


In [4]:
# ============================================================
# D) EMBEDDING, E) INDICIZZAZIONE FAISS
# ============================================================

def load_chunks() -> List[dict]:
    if not CHUNKS_PATH.exists():
        return []

    loaded = []
    with CHUNKS_PATH.open("r", encoding="utf-8") as handle:
        for line in handle:
            chunk = json.loads(line)
            text = (chunk.get("text") or "").strip()
            if len(text) >= 50:
                loaded.append(chunk)
    return loaded


def pdf_manifest(pdf_dir: Path = PDF_DIR) -> List[dict]:
    pdf_files = sorted(pdf_dir.rglob("*.pdf"))
    return [
        {
            "path": str(pdf_path.relative_to(pdf_dir)),
            "size": pdf_path.stat().st_size,
            "mtime_ns": pdf_path.stat().st_mtime_ns,
        }
        for pdf_path in pdf_files
    ]


def corpus_signature(manifest: List[dict]) -> str:
    payload = json.dumps(manifest, ensure_ascii=False, sort_keys=True)
    return hashlib.sha256(payload.encode("utf-8")).hexdigest()


def load_index_meta() -> dict:
    if not INDEX_META_PATH.exists():
        return {}
    try:
        with INDEX_META_PATH.open("r", encoding="utf-8") as handle:
            return json.load(handle)
    except json.JSONDecodeError:
        return {}


def index_rebuild_reason(current_manifest: List[dict]) -> str | None:
    if FORCE_REBUILD_INDEX:
        return "FORCE_REBUILD_INDEX=1"
    if not CHUNKS_PATH.exists():
        return f"file mancante: {CHUNKS_PATH}"
    if not FAISS_PATH.exists():
        return f"file mancante: {FAISS_PATH}"
    if not INDEX_META_PATH.exists():
        return f"file mancante: {INDEX_META_PATH}"

    meta = load_index_meta()
    current_signature = corpus_signature(current_manifest)
    if meta.get("pdf_dir") != str(PDF_DIR):
        return "PDF_DIR cambiata"
    if meta.get("model") != EMBEDDING_MODEL:
        return "modello embedding cambiato"
    if meta.get("pdf_manifest") != current_manifest:
        return "PDF aggiunti, rimossi o modificati"
    if meta.get("corpus_signature") != current_signature:
        return "firma corpus non coerente"
    return None


def build_faiss_index(chunks: List[dict], embedder: SentenceTransformer, manifest: List[dict]):
    texts = [chunk["text"] for chunk in chunks]
    if not texts:
        raise RuntimeError("Nessun testo valido per embedding.")

    embeddings = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=True)
    embeddings = np.asarray(embeddings, dtype="float32")
    if embeddings.ndim != 2 or embeddings.shape[0] == 0 or embeddings.shape[1] == 0:
        raise RuntimeError(f"Embeddings non validi: shape={embeddings.shape}")

    index = faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings)
    faiss.write_index(index, str(FAISS_PATH))

    meta = {
        "n_chunks": len(chunks),
        "model": EMBEDDING_MODEL,
        "dim": int(embeddings.shape[1]),
        "pdf_dir": str(PDF_DIR),
        "pdf_manifest": manifest,
        "corpus_signature": corpus_signature(manifest),
    }
    with INDEX_META_PATH.open("w", encoding="utf-8") as handle:
        json.dump(meta, handle, ensure_ascii=False, indent=2)

    print("Indice FAISS creato:", FAISS_PATH)
    print("Dimensione:", embeddings.shape[1], "Vettori:", index.ntotal)
    return index


def prepare_rag(force_rebuild: bool = FORCE_REBUILD_INDEX):
    print("Carico embedder...")
    embedder = SentenceTransformer(EMBEDDING_MODEL)

    global ACTIVE_CORPUS_SIGNATURE

    current_manifest = pdf_manifest(PDF_DIR)
    if not current_manifest:
        raise FileNotFoundError(f"Nessun PDF trovato in {PDF_DIR}")
    ACTIVE_CORPUS_SIGNATURE = corpus_signature(current_manifest)

    reason = "FORCE_REBUILD_INDEX=1" if force_rebuild else index_rebuild_reason(current_manifest)
    if reason:
        print("Ricreo chunks e indice FAISS dai PDF. Motivo:", reason)
        chunks = ingest_pdfs(PDF_DIR)
        index = build_faiss_index(chunks, embedder, current_manifest)
    else:
        chunks = load_chunks()
        if not chunks:
            print("chunks.jsonl vuoto: ricreo indice dai PDF...")
            chunks = ingest_pdfs(PDF_DIR)
            index = build_faiss_index(chunks, embedder, current_manifest)
        else:
            index = faiss.read_index(str(FAISS_PATH))
            print("Chunk caricati:", len(chunks))
            print("Indice FAISS caricato. n:", index.ntotal)

    if index.ntotal != len(chunks):
        raise RuntimeError(f"Indice/chunks non coerenti: index.ntotal={index.ntotal}, chunks={len(chunks)}")

    return chunks, embedder, index


chunks, embedder, index = prepare_rag()
ltm = VectorMemory(embedder)


Carico embedder...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Chunk caricati: 3999
Indice FAISS caricato. n: 3999


In [5]:
# ============================================================
# F) RETRIEVAL, G) BUILD CONTEXT
# ============================================================

def chunk_belongs_to_documents(chunk: dict, document_ids: List[str] | None) -> bool:
    if not document_ids:
        return True
    source = str(chunk.get("source") or "")
    chunk_id = str(chunk.get("id") or "")
    return any(source == doc_id or chunk_id.startswith(f"{doc_id}::") for doc_id in document_ids)


def retrieve(query: str, top_k: int = TOP_K, document_ids: List[str] | None = None) -> List[dict]:
    query = (query or "").strip()
    if not query:
        return []

    qv = embedder.encode([query], normalize_embeddings=True)
    qv = np.asarray(qv, dtype="float32").reshape(1, -1)

    if qv.shape[1] != index.d:
        raise ValueError(
            f"Dimension mismatch: qv dim={qv.shape[1]} vs index.d={index.d}. "
            "Indice FAISS e/o modello embedding non coerenti."
        )

    filtered_ids = [doc_id for doc_id in (document_ids or []) if doc_id]
    search_k = len(chunks) if filtered_ids else min(top_k, len(chunks))
    scores, ids = index.search(qv, search_k)

    results = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        item = dict(chunks[int(idx)])
        if not chunk_belongs_to_documents(item, filtered_ids):
            continue
        item["score"] = float(score)
        results.append(item)
        if len(results) >= top_k:
            break
    return results


def build_context(results: List[dict], max_chars: int = 4500) -> str:
    parts = []
    total = 0
    for result in results:
        header = (
            f"[Fonte: {result['source']} | Pagina: {result['page']} | "
            f"Chunk: {result['chunk_index']} | score: {result['score']:.3f}]"
        )
        snippet = header + "\n" + result["text"].strip()
        if total + len(snippet) > max_chars:
            break
        parts.append(snippet)
        total += len(snippet) + 2
    return "\n\n---\n\n".join(parts)


def preview_retrieval(question: str, top_k: int = TOP_K) -> List[dict]:
    hits = retrieve(question, top_k=top_k)
    print("\nTOP HITS:")
    for hit in hits:
        print(hit["id"], hit["score"])
    print("\nCONTESTO PREPARATO:\n")
    print(build_context(hits))
    return hits


In [6]:
# ============================================================
# H) GENERATION LLM - PROMPT XRSUPP SENZA BACKEND API
# ============================================================

QA_PROMPT = """Sei un assistente virtuale esperto della conoscenza e conrtesto dei pdf.

Personalita e stile:
- Rispondi sempre in italiano.
- Mantieni un tono amichevole, naturale, professionale e disponibile.
- Se l'utente saluta, ringrazia, chiede come stai, fa conversazione generica o pone una domanda amichevole, rispondi in modo cordiale e umano, senza forzare riferimenti tecnici.
- Non essere freddo o meccanico: usa frasi semplici, dirette e collaborative.
- Evita risposte troppo lunghe quando la domanda e semplice.
- Quando la domanda e tecnica, resta preciso, ordinato e concreto.

Uso del contesto:
- Usa il CONTEXT come fonte principale quando contiene informazioni rilevanti.
- Se il CONTEXT contiene fonti, pagine o chunk utili, integra la risposta con quelle informazioni.
- Se il CONTEXT non contiene abbastanza informazioni per rispondere completamente, non dire in modo esplicito e secco che l'informazione non e presente nel contesto.
- In assenza di informazioni sufficienti nel CONTEXT, costruisci comunque una risposta utile basandoti sulla tua conoscenza generale, mantenendo un tono prudente e professionale.
- Quando usi conoscenza generale non supportata direttamente dal CONTEXT, evita affermazioni troppo assolute e formula la risposta come indicazione generale.
- Non inventare nomi di schermate, menu, percorsi, campi o configurazioni specifiche se non sono nel CONTEXT o non sono ragionevolmente noti.
- Se una risposta richiede dati specifici dell'ambiente del cliente, spiega in modo amichevole cosa verificare o quali informazioni servirebbero.

Qualita della risposta:
- Dai prima la risposta pratica, poi eventuali dettagli.
- Per procedure operative, usa passaggi chiari e numerati.
- Per concetti o definizioni, usa una spiegazione breve seguita da esempi se utili.
- Per domande ambigue, rispondi con l'interpretazione piu probabile e, se serve, chiedi una precisazione alla fine.
- Se l'utente chiede un confronto, struttura la risposta in punti o tabella testuale semplice.
- Se l'utente chiede una configurazione NetSuite, evidenzia prerequisiti, percorso operativo e risultato atteso quando possibile.
- Mantieni risposte concise ma complete: evita testo superfluo, ma non omettere passaggi importanti.

Citazioni e fonti:
- Cita le fonti quando nel CONTEXT sono disponibili riferimenti a Fonte, Pagina o Chunk.
- Non citare fonti inesistenti.
- Se stai rispondendo a una domanda amichevole o generale non tecnica, non serve citare fonti.

MEMORIA BREVE:
{stm}

MEMORIA LUNGA RILEVANTE:
{ltm}

CONTEXT:
{context}

DOMANDA:
{question}

RISPOSTA (in italiano, chiara, strutturata e professionale):"""


def build_prompt(question: str, context: str, memory_hits: List[dict]) -> str:
    return QA_PROMPT.format(
        stm=stm_text(),
        ltm=ltm_text(memory_hits),
        context=context,
        question=question,
    )


def call_llm(prompt: str) -> str:
    client = OpenAI(api_key=OPENAI_API_KEY, base_url=LLM_BASE_URL)
    response = client.chat.completions.create(
        model=LLM_MODEL,
        temperature=0,
        messages=[{"role": "user", "content": prompt}],
    )
    return (response.choices[0].message.content or "").strip()


def ask(question: str, top_k: int = TOP_K, document_ids: List[str] | None = None, save_memory: bool = True) -> Dict[str, object]:
    hits = retrieve(question, top_k=top_k, document_ids=document_ids)
    context = build_context(hits)
    memory_hits = ltm.search(SESSION_ID, question, top_k=4)
    prompt = build_prompt(question, context, memory_hits)
    answer = call_llm(prompt)

    stm_add("user", question)
    stm_add("assistant", answer)
    if save_memory:
        ltm.add_turn(SESSION_ID, question, answer)

    print("\n=== RISPOSTA ===\n")
    print(answer)
    print("\n=== FONTI RECUPERATE ===")
    for hit in hits:
        print(f"- {hit['source']} | pagina {hit['page']} | chunk {hit['chunk_index']} | score {hit['score']:.3f}")

    return {
        "question": question,
        "answer": answer,
        "hits": hits,
        "context": context,
        "memory_hits": memory_hits,
        "prompt": prompt,
    }


def chat_loop() -> None:
    print("Chat RAG pronta. Scrivi una domanda oppure 'exit' per uscire.")
    while True:
        question = input("\nDomanda> ").strip()
        if question.lower() in {"exit", "quit", "q"}:
            break
        if question:
            ask(question)


In [7]:
# ============================================================
# ESECUZIONE BATCH DA TERMINALE
# ============================================================

question = os.getenv("RAG_QUESTION", "").strip()

if question:
    result = ask(question)
else:
    print("Indice pronto. Imposta RAG_QUESTION per esecuzione batch oppure usa ask('...') da notebook.")
    print("Esempio: ask('Come si creano nuovi record CRM in NetSuite?')")


Indice pronto. Imposta RAG_QUESTION per esecuzione batch oppure usa ask('...') da notebook.
Esempio: ask('Come si creano nuovi record CRM in NetSuite?')


In [12]:
# ============================================================
# I) DOMANDA MANUALE DA NOTEBOOK
# ============================================================

#REACH STACKER Serie F500H-RS

# Scrivi qui la domanda e poi esegui questa cella.
DOMANDA = """
come verifico il corretto funzionamento della pompa idraulica?
""".strip()

if not DOMANDA or DOMANDA == "Scrivi qui la tua domanda...":
    raise ValueError("Scrivi una domanda nella variabile DOMANDA prima di eseguire la cella.")

risultato = ask(DOMANDA)
risposta_llm = risultato["answer"]



=== RISPOSTA ===

Ciao! Sono felice di aiutarti con la tua domanda. Per verificare il corretto funzionamento della pompa idraulica, ti suggerisco di seguire questi passaggi:

1.  **Controlla la pressione**: Verifica che la pressione dell'impianto idraulico sia all'interno dei limiti stabiliti dal manuale del motore.
2.  **Ascolta il rumore**: Ascolta il rumore della pompa idraulica durante l'avvio e il cambio di carico. Se il rumore è anomalo o troppo forte, potrebbe indicare una carenza di lubrificazione.
3.  **Verifica la temperatura**: Controlla la temperatura dell'impianto idraulico per assicurarti che non sia troppo alta. Una temperatura eccessiva può indicare problemi con il sistema.
4.  **Controlla i livelli dei fluidi**: Verifica che i livelli dei fluidi siano corretti e che non ci siano perdite dall'impianto idraulico.

Per ulteriori informazioni, ti consiglio di consultare il manuale del motore Cummins "QSG12 / X1" e di controllare se ci sono specifiche indicazioni per la tu

In [ ]:
# ============================================================
# J) ANALISI COMPARATIVA SEMANTICA RISPOSTA LLM vs RISPOSTA REALE
# ============================================================

# Scrivi qui la risposta corretta/attesa.
RISPOSTA_REALE = """

""".strip()


def _normalizza_testo(testo: str) -> str:
    testo = (testo or "").lower()
    testo = re.sub(r"[^a-zàèéìòù0-9\s]", " ", testo)
    return re.sub(r"\s+", " ", testo).strip()


def _token_significativi(testo: str) -> set[str]:
    stopwords = {
        "a", "ad", "al", "alla", "allo", "ai", "agli", "alle", "anche", "che", "con",
        "da", "dal", "dalla", "dei", "del", "della", "di", "e", "gli", "i", "il",
        "in", "la", "le", "lo", "ma", "nel", "nella", "o", "per", "piu", "più",
        "sono", "su", "tra", "un", "una", "uno", "è", "essere", "come", "non",
        "si", "solo", "sul", "sulla", "questo", "questa", "quello", "quella",
    }
    return {tok for tok in _normalizza_testo(testo).split() if len(tok) >= 4 and tok not in stopwords}


def _cosine_semantica(testo_a: str, testo_b: str) -> float:
    vettori = embedder.encode([testo_a, testo_b], normalize_embeddings=True)
    vettori = np.asarray(vettori, dtype="float32")
    return float(np.dot(vettori[0], vettori[1]))


def analisi_comparativa_semantica(risposta_llm_param: str, risposta_reale_param: str, soglia_ok: float = 0.75) -> dict:
    """Confronta semanticamente risposta LLM e risposta reale attesa."""
    risposta_llm_param = (risposta_llm_param or "").strip()
    risposta_reale_param = (risposta_reale_param or "").strip()

    if not risposta_llm_param:
        raise ValueError("La risposta LLM passata come parametro è vuota.")
    if not risposta_reale_param or risposta_reale_param == "Scrivi qui la risposta reale...":
        raise ValueError("Compila RISPOSTA_REALE con la risposta corretta/attesa prima di eseguire la cella.")

    similarita = _cosine_semantica(risposta_llm_param, risposta_reale_param)

    token_llm = _token_significativi(risposta_llm_param)
    token_reali = _token_significativi(risposta_reale_param)
    concetti_comuni = sorted(token_llm & token_reali)
    concetti_mancanti = sorted(token_reali - token_llm)
    concetti_extra = sorted(token_llm - token_reali)

    copertura_lessicale = (len(concetti_comuni) / len(token_reali)) if token_reali else 0.0
    esito = "OK" if similarita >= soglia_ok else "DA VERIFICARE"

    report = {
        "similarita_semantica": round(similarita, 3),
        "copertura_concetti_risposta_reale": round(copertura_lessicale, 3),
        "esito": esito,
        "concetti_comuni": concetti_comuni[:30],
        "concetti_mancanti_nella_risposta_llm": concetti_mancanti[:30],
        "concetti_extra_nella_risposta_llm": concetti_extra[:30],
    }

    print("=== ANALISI COMPARATIVA SEMANTICA ===")
    print(f"Similarità semantica: {report['similarita_semantica']:.3f}")
    print(f"Copertura concetti risposta reale: {report['copertura_concetti_risposta_reale']:.3f}")
    print("Esito:", report["esito"])
    print("\nConcetti comuni:", ", ".join(report["concetti_comuni"]) or "nessuno")
    print("\nConcetti della risposta reale mancanti nella risposta LLM:", ", ".join(report["concetti_mancanti_nella_risposta_llm"]) or "nessuno")
    print("\nConcetti extra nella risposta LLM:", ", ".join(report["concetti_extra_nella_risposta_llm"]) or "nessuno")

    return report


analisi_semantica = analisi_comparativa_semantica(
    risposta_llm_param=risposta_llm,
    risposta_reale_param=RISPOSTA_REALE,
)
